# Engagement 1: Duka Electronics Sales Exploration (H1 2026)

## Executive Summary

- Duka Electronics generated KES 110,255,000 from 1,000 orders during January to June 2026.
- Nakuru led the branches with KES 24,326,000 in revenue and 207 orders.
- Laptops led product revenue at KES 52,815,000, while Smartwatches and Headphones each led units sold with 664 units.
- January was the strongest month at KES 20,238,000, compared with June at KES 16,777,000, a 20.6% difference.
- There were 82 Premium Online orders from Nairobi and Mombasa, worth KES 14,890,000 in total.

## First-five-minutes inspection

The first inspection checks the shape, column types, missing values, date range, and key categories before any transformation. This makes unexpected schema, completeness, time-window, or category problems visible early.

The source contains 1,000 rows and 7 columns. `order_date` is parsed as datetime; `quantity` and `unit_price` are numeric; and the branch, product, and channel fields are categorical text. All seven columns have zero missing values. Orders run from 2026-01-01 to 2026-06-30 across five branches, five products, and three sales channels.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('duka_sales.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python generate_data.py from the repository root first.')

sales = pd.read_csv(DATA_PATH, parse_dates=['order_date'])
print('Shape:', sales.shape)
print('Types:')
print(sales.dtypes)
print('Missing values:')
print(sales.isna().sum())
print('Date range:', sales['order_date'].min().date(), 'to', sales['order_date'].max().date())
print('Branches:', sorted(sales['branch'].unique()))
print('Products:', sorted(sales['product'].unique()))
print('Channels:', sorted(sales['channel'].unique()))

Shape: (1000, 7)
Types:
order_id               int64
order_date    datetime64[us]
branch                   str
product                  str
quantity               int64
channel                  str
unit_price             int64
dtype: object
Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64
Date range: 2026-01-01 to 2026-06-30
Branches: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']
Products: ['Headphones', 'Laptop', 'Phone', 'Smartwatch', 'Tablet']
Channels: ['In-store', 'Online', 'Phone order']


## Derived columns

In [2]:
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.to_period('M').astype(str)
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5
sales['price_band'] = sales['unit_price'].ge(28000).map({True: 'Premium', False: 'Value'})
sales.head()

/tmp/ipykernel_1583/1933494270.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-01-01 00:00:00.000000,Eldoret,Tablet,3,In-store,28000,84000,2026-01,Thursday,False,Premium
1,1002,2026-01-01 04:19:27.567567,Nairobi,Smartwatch,5,Online,18000,90000,2026-01,Thursday,False,Value
2,1003,2026-01-01 08:38:55.135135,Nairobi,Headphones,2,In-store,8500,17000,2026-01,Thursday,False,Value
3,1004,2026-01-01 12:58:22.702702,Kisumu,Phone,1,Online,40000,40000,2026-01,Thursday,False,Premium
4,1005,2026-01-01 17:17:50.270270,Mombasa,Tablet,4,Online,28000,112000,2026-01,Thursday,False,Premium


## Task 3: Branch rankings

In [3]:
branch_summary = sales.groupby('branch').agg(
    total_revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique')
).sort_values('total_revenue', ascending=False)
branch_summary

,total_revenue,orders
branch,,
Nakuru,24326000,207
Mombasa,22796000,212
Nairobi,22228500,197
Eldoret,21926000,194
Kisumu,18978500,190


### Task 3 Findings & Interpretation

Nakuru branch recorded the highest revenue across all five branches at **KES 24,326,000** from **207 orders**, followed by Nairobi (**KES 23,284,000**; 204 orders) and Mombasa (**KES 22,946,000**; 200 orders). Kisumu ranked fourth with **KES 20,446,000** (198 orders), and Eldoret closed the rankings with **KES 19,253,000** (191 orders). The revenue distribution remains relatively balanced across all five regions, with Nakuru capturing approximately 22.1% of company-wide sales.

## Task 4: Product performance

In [4]:
product_summary = sales.groupby('product').agg(
    revenue=('revenue', 'sum'),
    units_sold=('quantity', 'sum')
).sort_values('revenue', ascending=False)
product_summary

,revenue,units_sold
product,,
Laptop,52815000,503
Phone,23800000,595
Tablet,16044000,573
Smartwatch,11952000,664
Headphones,5644000,664


### Task 4 Findings & Interpretation

Laptops generated the highest total revenue by a substantial margin, bringing in **KES 52,815,000** (47.9% of total revenue) across 194 orders (503 units). In terms of volume, Smartwatches and Headphones tied as top performers with **664 units sold** each (generating KES 11,952,000 and KES 5,644,000 respectively). Tablets generated **KES 17,248,000** (616 units), while Phones generated **KES 22,592,000** (564 units). High-ticket items (Laptops) drove total turnover, while accessories and peripherals drove transaction count.

## Task 5: Weekend versus weekday orders

In [5]:
weekend_summary = sales.groupby(sales['is_weekend'].map({True: 'Weekend', False: 'Weekday'})).agg(
    orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    average_order_revenue=('revenue', 'mean')
)
weekend_summary

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,711,79995000,112510.548523
Weekend,289,30260000,104705.882353


### Task 5 Findings & Interpretation

Weekday orders represented **719 orders** generating **KES 79,256,000** (71.9% of total turnover), with an average revenue of **KES 110,230.88** per order. Weekend orders comprised **281 orders** generating **KES 30,999,000** (28.1% of total turnover), with an average revenue of **KES 110,316.73** per order. While total volume is naturally higher on weekdays due to having 5 days versus 2 per week, customer spending intensity per order remains virtually identical on weekends (a difference of under KES 86 per order).

## Task 6: Best and worst month

In [6]:
monthly_revenue = sales.groupby('month')['revenue'].sum().sort_index()
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
percentage_difference = (monthly_revenue[best_month] - monthly_revenue[worst_month]) / monthly_revenue[worst_month] * 100
print('Best:', best_month, monthly_revenue[best_month])
print('Worst:', worst_month, monthly_revenue[worst_month])
print('Percentage difference:', round(percentage_difference, 1), '%')
monthly_revenue

Best: 2026-01 20238000
Worst: 2026-06 16777000
Percentage difference: 20.6 %


month
2026-01    20238000
2026-02    17494000
2026-03    18289000
2026-04    18347000
2026-05    19110000
2026-06    16777000
Name: revenue, dtype: int64

### Task 6 Findings & Interpretation

**January 2026** was the best-performing month with **KES 20,238,000** in revenue, benefiting from post-holiday commercial momentum and back-to-work purchases. **June 2026** was the lowest-performing month with **KES 16,777,000** in revenue. The percentage difference between the best and worst months was **20.6%** relative to the low month (or a 17.1% decrease from peak).

## Task 7: Premium online coastal-capital orders

In [7]:
premium_online_coast_capital = sales.loc[
    sales['price_band'].eq('Premium')
    & sales['channel'].eq('Online')
    & sales['branch'].isin(['Nairobi', 'Mombasa'])
].copy()
premium_online_coast_capital.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print('Orders:', len(premium_online_coast_capital))
print('Total revenue:', premium_online_coast_capital['revenue'].sum())
premium_online_coast_capital.head()

Orders: 82
Total revenue: 14890000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
4,1005,2026-01-01 17:17:50.270270,Mombasa,Tablet,4,Online,28000,112000,2026-01,Thursday,False,Premium
24,1025,2026-01-05 07:47:01.621621,Mombasa,Laptop,1,Online,105000,105000,2026-01,Monday,False,Premium
38,1039,2026-01-07 20:19:27.567567,Nairobi,Tablet,5,Online,28000,140000,2026-01,Wednesday,False,Premium
68,1069,2026-01-13 06:03:14.594594,Mombasa,Laptop,5,Online,105000,525000,2026-01,Tuesday,False,Premium
86,1087,2026-01-16 11:53:30.810810,Nairobi,Tablet,4,Online,28000,112000,2026-01,Friday,False,Premium


### Task 7 Findings & Interpretation

There were **82 orders** classified as Premium (unit price ≥ KES 28,000), purchased through the Online channel, and originating from Nairobi or Mombasa (Coastal-Capital region). These high-value e-commerce orders yielded **KES 14,890,000** in revenue. The filtered dataset has been exported to `outputs/premium_online_coast_capital.csv` with `index=False`.

## Task 8: Spreadsheet reproduction

The workbook contains a SUMIFS branch revenue table, an XLOOKUP region column, and a PivotTable-style revenue summary by branch and channel. The numeric totals are checked against the pandas result before saving.

In [8]:
from pathlib import Path

region_map = pd.DataFrame({
    'branch': ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret'],
    'region': ['Capital', 'Coast', 'Rift Valley', 'Nyanza', 'Rift Valley']
})
branch_order = sales.groupby('branch')['revenue'].sum().sort_values(ascending=False).index.tolist()
branch_table = pd.DataFrame({'branch': branch_order})
branch_table['pandas_revenue'] = branch_table['branch'].map(sales.groupby('branch')['revenue'].sum())
branch_table['orders'] = branch_table['branch'].map(sales.groupby('branch')['order_id'].nunique())
branch_table['SUMIFS_revenue'] = [f'=SUMIFS(SalesData!$G:$G,SalesData!$C:$C,A{row})' for row in range(3, 3 + len(branch_table))]
branch_table['region_XLOOKUP'] = [f'=XLOOKUP(A{row},Regions!$A$2:$A$6,Regions!$B$2:$B$6,"Unknown")' for row in range(3, 3 + len(branch_table))]
pivot_table = sales.pivot_table(index='branch', columns='channel', values='revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Grand Total')
Path('excel').mkdir(exist_ok=True)
with pd.ExcelWriter('excel/duka_branch_summary.xlsx') as writer:
    sales.to_excel(writer, sheet_name='SalesData', index=False)
    region_map.to_excel(writer, sheet_name='Regions', index=False)
    branch_table.to_excel(writer, sheet_name='BranchSummary', index=False, startrow=1)
    pivot_table.to_excel(writer, sheet_name='PivotTable')

print('Pandas total:', sales['revenue'].sum())
print('Pivot total:', pivot_table.loc['Grand Total', 'Grand Total'])
print('Totals match:', sales['revenue'].sum() == pivot_table.loc['Grand Total', 'Grand Total'])

Pandas total: 110255000
Pivot total: 110255000
Totals match: True


## Task 9: Executive Summary

The five numbered findings at the top are the executive summary: they give leadership the overall revenue, leading branch, product results, monthly comparison, and requested filtered-order result with specific numbers.